In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

random.seed(42)
np.random.seed(42)

In [2]:
# Experiment configuration
N_USERS = 25000                          
START_DATE = datetime(2024, 1, 1)
END_DATE = datetime(2024, 6, 30)        

# Groups
GROUPS = ['Control', 'Treatment']

# Device types
DEVICES = ['Mobile', 'Desktop', 'Tablet']
DEVICE_WEIGHTS = [0.72, 0.22, 0.06]

# City tiers
CITY_TIERS = ['Tier 1', 'Tier 2', 'Tier 3']
CITY_WEIGHTS = [0.35, 0.42, 0.23]

# Age groups
AGE_GROUPS = ['18-24', '25-34', '35-44', '45+']
AGE_WEIGHTS = [0.28, 0.38, 0.22, 0.12]

# Payment methods (only assigned to purchasers later)
PAYMENT_METHODS = ['UPI', 'Credit Card', 'Debit Card', 'COD']
PAYMENT_WEIGHTS_CONTROL   = [0.38, 0.22, 0.18, 0.22]   # control: UPI not dominant
PAYMENT_WEIGHTS_TREATMENT = [0.61, 0.18, 0.13, 0.08]   # treatment: nudge pushes UPI adoption

In [3]:
# Generate user IDs
user_ids = [f'USER_{str(i).zfill(5)}' for i in range(1, N_USERS + 1)]

# Assign groups (50/50 split)
groups = ['Control'] * (N_USERS // 2) + ['Treatment'] * (N_USERS // 2)
random.shuffle(groups)

# Assign devices, cities, age groups
devices    = random.choices(DEVICES, weights=DEVICE_WEIGHTS, k=N_USERS)
city_tiers = random.choices(CITY_TIERS, weights=CITY_WEIGHTS, k=N_USERS)
age_groups = random.choices(AGE_GROUPS, weights=AGE_WEIGHTS, k=N_USERS)

# Returning vs new customer (60% new, 40% returning — realistic for a growing D2C brand)
returning_customer = random.choices([0, 1], weights=[0.60, 0.40], k=N_USERS)

# Session duration in minutes (treatment group browses slightly longer due to nudge)
session_duration = []
for i in range(N_USERS):
    if groups[i] == 'Treatment':
        mins = round(np.random.normal(loc=6.8, scale=2.5), 1)
    else:
        mins = round(np.random.normal(loc=5.9, scale=2.5), 1)
    session_duration.append(max(0.5, mins))   # minimum 30 seconds

# Pages visited before checkout
pages_visited = []
for i in range(N_USERS):
    base = 4.2 if groups[i] == 'Treatment' else 3.8
    pages = int(np.random.normal(loc=base, scale=1.5))
    pages_visited.append(max(1, pages))

# Generate session dates
def random_date(start, end):
    delta = end - start
    return start + timedelta(days=random.randint(0, delta.days))

session_dates = [random_date(START_DATE, END_DATE) for _ in range(N_USERS)]

In [4]:
# Conversion probabilities - Control group (no nudge)
control_probs = {
    'add_to_cart':   0.60,   # 60% of visitors add to cart
    'reach_checkout': 0.55,  # 55% of those reach checkout
    'purchase':      0.45    # 45% of those actually purchase
}

# Conversion probabilities - Treatment group (with UPI nudge)
# The nudge improves checkout and purchase rates slightly
treatment_probs = {
    'add_to_cart':   0.61,   # barely changes (nudge shown later in funnel)
    'reach_checkout': 0.58,  # slightly better - nudge creates curiosity
    'purchase':      0.52    # meaningfully better - UPI cashback seals the deal
}

# Now simulate each user's journey through the funnel
add_to_cart_list = []
reach_checkout_list = []
purchase_list = []

for i in range(N_USERS):
    group = groups[i]
    probs = control_probs if group == 'Control' else treatment_probs

    # Step 1: Did user add to cart?
    atc = int(random.random() < probs['add_to_cart'])

    # Step 2: Did user reach checkout? (only possible if they added to cart)
    if atc == 1:
        rc = int(random.random() < probs['reach_checkout'])
    else:
        rc = 0

    # Step 3: Did user purchase? (only possible if they reached checkout)
    if rc == 1:
        pur = int(random.random() < probs['purchase'])
    else:
        pur = 0

    add_to_cart_list.append(atc)
    reach_checkout_list.append(rc)
    purchase_list.append(pur)

In [6]:
order_values   = []
payment_method = []

for i in range(N_USERS):
    if purchase_list[i] == 1:
        # Returning customers tend to spend more (they trust the brand)
        if returning_customer[i] == 1:
            value = round(np.random.normal(loc=980, scale=240), 2)
        else:
            value = round(np.random.normal(loc=780, scale=210), 2)
        value = max(199, value)
        order_values.append(value)

        # Assign payment method based on group
        if groups[i] == 'Treatment':
            pm = random.choices(PAYMENT_METHODS, weights=PAYMENT_WEIGHTS_TREATMENT)[0]
        else:
            pm = random.choices(PAYMENT_METHODS, weights=PAYMENT_WEIGHTS_CONTROL)[0]
        payment_method.append(pm)
    else:
        order_values.append(0)
        payment_method.append('None')

In [12]:
df = pd.DataFrame({
    'user_id':              user_ids,
    'session_date':         session_dates,
    'group':                groups,
    'device':               devices,
    'city_tier':            city_tiers,
    'age_group':            age_groups,
    'returning_customer':   returning_customer,
    'session_duration_mins':session_duration,
    'pages_visited':        pages_visited,
    'add_to_cart':          add_to_cart_list,
    'reached_checkout':     reach_checkout_list,
    'purchased':            purchase_list,
    'payment_method':       payment_method,
    'order_value_inr':      order_values
})

df = df.sort_values('session_date').reset_index(drop=True)

print(f"Dataset shape: {df.shape}")
df.head(10)

Dataset shape: (25000, 14)


,user_id,session_date,group,device,city_tier,age_group,returning_customer,session_duration_mins,pages_visited,add_to_cart,reached_checkout,purchased,payment_method,order_value_inr
0,USER_18835,2024-01-01,Treatment,Mobile,Tier 3,18-24,0,6.7,1,1,0,0,None,0.00
1,USER_03056,2024-01-01,Treatment,Mobile,Tier 2,18-24,0,4.7,4,0,0,0,None,0.00
2,USER_17137,2024-01-01,Control,Mobile,Tier 1,25-34,0,7.5,5,0,0,0,None,0.00
3,USER_12418,2024-01-01,Treatment,Mobile,Tier 3,25-34,1,7.2,3,0,0,0,None,0.00
4,USER_20209,2024-01-01,Treatment,Mobile,Tier 3,18-24,1,1.6,2,1,0,0,None,0.00
5,USER_13324,2024-01-01,Control,Desktop,Tier 1,18-24,0,5.9,4,1,0,0,None,0.00
6,USER_13288,2024-01-01,Control,Tablet,Tier 1,25-34,1,6.5,2,1,0,0,None,0.00
7,USER_21391,2024-01-01,Control,Mobile,Tier 1,25-34,1,8.6,4,0,0,0,None,0.00
8,USER_24632,2024-01-01,Control,Mobile,Tier 2,25-34,0,8.3,1,1,0,0,None,0.00
9,USER_12488,2024-01-01,Control,Mobile,Tier 2,35-44,0,9.3,3,1,1,1,Debit Card,928.49


In [8]:
df.to_csv('upi_ab_test_data.csv', index=False)

print("✅ Dataset saved!")
print(f"\n{'='*45}")
print(f"  EXPERIMENT SUMMARY")
print(f"{'='*45}")
print(f"  Total users      : {len(df):,}")
print(f"  Date range       : Jan 2024 – Jun 2024")
print(f"  Control group    : {len(df[df['group']=='Control']):,}")
print(f"  Treatment group  : {len(df[df['group']=='Treatment']):,}")
print(f"{'='*45}")

for grp in ['Control', 'Treatment']:
    sub = df[df['group'] == grp]
    purchasers = sub[sub['purchased'] == 1]
    print(f"\n  {grp} group")
    print(f"  Purchase rate    : {sub['purchased'].mean():.2%}")
    print(f"  Avg order value  : ₹{purchasers['order_value_inr'].mean():,.0f}")
    print(f"  Avg session (min): {sub['session_duration_mins'].mean():.1f}")
    print(f"  UPI usage        : {(purchasers['payment_method']=='UPI').mean():.2%}")

✅ Dataset saved!

  EXPERIMENT SUMMARY
  Total users      : 25,000
  Date range       : Jan 2024 – Jun 2024
  Control group    : 12,500
  Treatment group  : 12,500

  Control group
  Purchase rate    : 15.00%
  Avg order value  : ₹857
  Avg session (min): 5.9
  UPI usage        : 37.01%

  Treatment group
  Purchase rate    : 18.26%
  Avg order value  : ₹862
  Avg session (min): 6.8
  UPI usage        : 62.02%


In [9]:
import sqlite3

# Create an in-memory SQLite database and load your dataset into it
conn = sqlite3.connect('upi_ab_test.db')

# Load the CSV into SQL as a table called 'experiment'
df.to_sql('experiment', conn, if_exists='replace', index=False)

print("✅ Database ready! Table 'experiment' loaded with 25,000 rows.")

✅ Database ready! Table 'experiment' loaded with 25,000 rows.


In [10]:
# This function lets us run any SQL query and see results as a clean table
def run_query(sql):
    return pd.read_sql_query(sql, conn)

In [15]:
query1 = """
SELECT
    `group` AS group_name,
    COUNT(*) AS total_users,

    SUM(add_to_cart) AS added_to_cart,
    ROUND(100.0 * SUM(add_to_cart) / COUNT(*), 2) AS add_to_cart_rate,

    SUM(reached_checkout) AS reached_checkout,
    ROUND(100.0 * SUM(reached_checkout) / COUNT(*), 2) AS checkout_rate,

    SUM(purchased) AS total_purchases,
    ROUND(100.0 * SUM(purchased) / COUNT(*), 2) AS purchase_rate

FROM experiment
GROUP BY `group`
ORDER BY `group`
"""

result1 = run_query(query1)
print("📊 Query 1: Overall Funnel by Group")
print(result1.to_string(index=False))

📊 Query 1: Overall Funnel by Group
group_name  total_users  added_to_cart  add_to_cart_rate  reached_checkout  checkout_rate  total_purchases  purchase_rate
   Control        12500           7478             59.82              4147          33.18             1875          15.00
 Treatment        12500           7547             60.38              4339          34.71             2283          18.26


In [17]:
query2 = """
SELECT
    `group`,
    ROUND(100.0 * SUM(add_to_cart) / COUNT(*), 2)
        AS pct_added_to_cart,

    ROUND(100.0 * SUM(reached_checkout) / NULLIF(SUM(add_to_cart), 0), 2)
        AS pct_checkout_of_cart,

    ROUND(100.0 * SUM(purchased) / NULLIF(SUM(reached_checkout), 0), 2)
        AS pct_purchased_of_checkout,

    ROUND(100.0 * (1 - 1.0*SUM(add_to_cart)/COUNT(*)), 2)
        AS dropoff_before_cart,

    ROUND(100.0 * (SUM(add_to_cart) - SUM(reached_checkout)) / NULLIF(SUM(add_to_cart),0), 2)
        AS dropoff_cart_to_checkout,

    ROUND(100.0 * (SUM(reached_checkout) - SUM(purchased)) / NULLIF(SUM(reached_checkout),0), 2)
        AS dropoff_checkout_to_purchase

FROM experiment
GROUP BY `group`
"""

result2 = run_query(query2)
print("📊 Query 2: Drop-off at each funnel stage")
print(result2.to_string(index=False))

📊 Query 2: Drop-off at each funnel stage
    group  pct_added_to_cart  pct_checkout_of_cart  pct_purchased_of_checkout  dropoff_before_cart  dropoff_cart_to_checkout  dropoff_checkout_to_purchase
  Control              59.82                 55.46                      45.21                40.18                     44.54                         54.79
Treatment              60.38                 57.49                      52.62                39.62                     42.51                         47.38


In [18]:
query3 = """
SELECT
    city_tier,
    `group`,
    COUNT(*) AS users,
    ROUND(100.0 * SUM(purchased) / COUNT(*), 2) AS purchase_rate,
    ROUND(AVG(CASE WHEN purchased = 1 THEN order_value_inr END), 0) AS avg_order_value
FROM experiment
GROUP BY city_tier, `group`
ORDER BY city_tier, `group`
"""

result3 = run_query(query3)
print("📊 Query 3: Performance by city tier")
print(result3.to_string(index=False))

📊 Query 3: Performance by city tier
city_tier     group  users  purchase_rate  avg_order_value
   Tier 1   Control   4339          14.87            870.0
   Tier 1 Treatment   4406          17.64            860.0
   Tier 2   Control   5224          15.07            849.0
   Tier 2 Treatment   5236          18.47            874.0
   Tier 3   Control   2937          15.08            854.0
   Tier 3 Treatment   2858          18.86            844.0


In [19]:
query4 = """
SELECT
    device,
    `group`,
    COUNT(*) AS users,
    ROUND(100.0 * SUM(purchased) / COUNT(*), 2) AS purchase_rate,
    ROUND(AVG(session_duration_mins), 1) AS avg_session_mins
FROM experiment
GROUP BY device, `group`
ORDER BY device, `group`
"""

result4 = run_query(query4)
print("📊 Query 4: Performance by device")
print(result4.to_string(index=False))

📊 Query 4: Performance by device
 device     group  users  purchase_rate  avg_session_mins
Desktop   Control   2748          15.32               5.9
Desktop Treatment   2803          17.66               6.8
 Mobile   Control   9070          14.79               5.9
 Mobile Treatment   8934          18.49               6.8
 Tablet   Control    682          16.57               5.8
 Tablet Treatment    763          17.82               6.8


In [22]:
query5 = """
SELECT
    `group`,
    payment_method,
    COUNT(*) AS users,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY `group`), 2) AS pct_of_group
FROM experiment
WHERE purchased = 1
GROUP BY `group`, payment_method
ORDER BY `group`, pct_of_group DESC
"""

result5 = run_query(query5)
print("📊 Query 5: Payment method breakdown (purchasers only)")
print(result5.to_string(index=False))

📊 Query 5: Payment method breakdown (purchasers only)
    group payment_method  users  pct_of_group
  Control            UPI    694         37.01
  Control            COD    434         23.15
  Control    Credit Card    390         20.80
  Control     Debit Card    357         19.04
Treatment            UPI   1416         62.02
Treatment    Credit Card    411         18.00
Treatment     Debit Card    284         12.44
Treatment            COD    172          7.53


In [23]:
query6 = """
SELECT
    `group`,
    SUM(purchased) AS total_orders,
    ROUND(SUM(order_value_inr), 0) AS total_revenue_inr,
    ROUND(AVG(CASE WHEN purchased=1 THEN order_value_inr END), 0) AS avg_order_value,
    ROUND(SUM(order_value_inr) / COUNT(*), 0) AS revenue_per_visitor
FROM experiment
GROUP BY `group`
"""

result6 = run_query(query6)
print("📊 Query 6: Revenue summary")
print(result6.to_string(index=False))

📊 Query 6: Revenue summary
    group  total_orders  total_revenue_inr  avg_order_value  revenue_per_visitor
  Control          1875          1607441.0            857.0                129.0
Treatment          2283          1968237.0            862.0                157.0


In [24]:
from scipy import stats
import scipy.stats as st
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries loaded")

✅ Libraries loaded


In [27]:
# Pull key numbers from your dataset
control   = df[df['group'] == 'Control']
treatment = df[df['group'] == 'Treatment']

# Sample sizes
n_control   = len(control)
n_treatment = len(treatment)

# Number of purchases in each group
purchases_control   = control['purchased'].sum()
purchases_treatment = treatment['purchased'].sum()

# Conversion rates
cr_control   = purchases_control / n_control
cr_treatment = purchases_treatment / n_treatment

print(f"Control   → {purchases_control} purchases / {n_control} users = {cr_control:.4f} ({cr_control:.2%})")
print(f"Treatment → {purchases_treatment} purchases / {n_treatment} users = {cr_treatment:.4f} ({cr_treatment:.2%})")
print(f"\nObserved lift: {(cr_treatment - cr_control):.4f} ({(cr_treatment - cr_control):.2%})")

Control   → 1875 purchases / 12500 users = 0.1500 (15.00%)
Treatment → 2283 purchases / 12500 users = 0.1826 (18.26%)

Observed lift: 0.0326 (3.26%)


In [28]:
# Build a contingency table
# Format: [[converted, not converted], [converted, not converted]]

converted_control       = purchases_control
not_converted_control   = n_control - purchases_control
converted_treatment     = purchases_treatment
not_converted_treatment = n_treatment - purchases_treatment

contingency_table = [
    [converted_control,   not_converted_control],
    [converted_treatment, not_converted_treatment]
]

print("Contingency Table:")
print(f"                Purchased    Not Purchased")
print(f"Control         {converted_control:<12} {not_converted_control}")
print(f"Treatment       {converted_treatment:<12} {not_converted_treatment}")

# Run chi-square test
chi2, p_value, dof, expected = stats.chi2_contingency(contingency_table)

print(f"\n{'='*40}")
print(f"  CHI-SQUARE TEST RESULTS")
print(f"{'='*40}")
print(f"  Chi-square statistic : {chi2:.4f}")
print(f"  P-value              : {p_value:.6f}")
print(f"  Degrees of freedom   : {dof}")
print(f"{'='*40}")

if p_value < 0.05:
    print(f"\n  ✅ SIGNIFICANT — p={p_value:.6f} < 0.05")
    print(f"  The nudge effect is REAL, not random chance.")
else:
    print(f"\n  ❌ NOT SIGNIFICANT — p={p_value:.6f} > 0.05")
    print(f"  Cannot conclude the nudge made a difference.")

Contingency Table:
                Purchased    Not Purchased
Control         1875         10625
Treatment       2283         10217

  CHI-SQUARE TEST RESULTS
  Chi-square statistic : 47.7865
  P-value              : 0.000000
  Degrees of freedom   : 1

  ✅ SIGNIFICANT — p=0.000000 < 0.05
  The nudge effect is REAL, not random chance.


In [29]:
import math

def confidence_interval(purchases, n, confidence=0.95):
    p = purchases / n
    z = st.norm.ppf((1 + confidence) / 2)   # z-score for 95% = 1.96
    margin = z * math.sqrt(p * (1 - p) / n)
    return (p - margin, p + margin)

ci_control   = confidence_interval(purchases_control, n_control)
ci_treatment = confidence_interval(purchases_treatment, n_treatment)

print(f"{'='*50}")
print(f"  CONFIDENCE INTERVALS (95%)")
print(f"{'='*50}")
print(f"  Control   : {cr_control:.2%}  [{ci_control[0]:.2%}  —  {ci_control[1]:.2%}]")
print(f"  Treatment : {cr_treatment:.2%}  [{ci_treatment[0]:.2%}  —  {ci_treatment[1]:.2%}]")
print(f"{'='*50}")

# Check if intervals overlap
if ci_treatment[0] > ci_control[1]:
    print(f"\n  ✅ Intervals do NOT overlap — strong evidence")
    print(f"  the treatment effect is real.")
else:
    print(f"\n  ⚠️  Intervals overlap — interpret with caution.")

  CONFIDENCE INTERVALS (95%)
  Control   : 15.00%  [14.37%  —  15.63%]
  Treatment : 18.26%  [17.59%  —  18.94%]

  ✅ Intervals do NOT overlap — strong evidence
  the treatment effect is real.


In [30]:
# Absolute lift
absolute_lift = cr_treatment - cr_control

# Relative lift (% improvement over baseline)
relative_lift = (cr_treatment - cr_control) / cr_control

print(f"{'='*50}")
print(f"  LIFT ANALYSIS")
print(f"{'='*50}")
print(f"  Absolute lift  : +{absolute_lift:.2%}")
print(f"  Relative lift  : +{relative_lift:.2%}")
print(f"{'='*50}")
print(f"""
  What this means:
  The UPI nudge improved conversion by {absolute_lift:.2%}
  in absolute terms — or a {relative_lift:.2%} relative
  improvement over the control baseline.

  In industry, a relative lift above 5% is
  considered meaningful for checkout experiments.
  Ours is {relative_lift:.2%} — well above threshold.
""")

  LIFT ANALYSIS
  Absolute lift  : +3.26%
  Relative lift  : +21.76%

  What this means:
  The UPI nudge improved conversion by 3.26%
  in absolute terms — or a 21.76% relative
  improvement over the control baseline.

  In industry, a relative lift above 5% is
  considered meaningful for checkout experiments.
  Ours is 21.76% — well above threshold.



In [31]:
from statsmodels.stats.power import NormalIndPower

# Effect size (Cohen's h for proportions)
effect_size = 2 * math.asin(math.sqrt(cr_treatment)) - \
              2 * math.asin(math.sqrt(cr_control))

# Power analysis
analysis = NormalIndPower()
required_n = analysis.solve_power(
    effect_size=abs(effect_size),
    power=0.80,           # 80% power — industry standard
    alpha=0.05,           # 5% significance level
    alternative='two-sided'
)

print(f"{'='*50}")
print(f"  POWER ANALYSIS")
print(f"{'='*50}")
print(f"  Effect size (Cohen's h) : {abs(effect_size):.4f}")
print(f"  Required sample size    : {int(required_n):,} per group")
print(f"  Actual sample size      : {n_control:,} per group")
print(f"{'='*50}")

if n_control >= required_n:
    print(f"\n  ✅ Our sample size is SUFFICIENT.")
    print(f"  We had {n_control:,} users vs the {int(required_n):,} required.")
    print(f"  Our test has adequate statistical power.")
else:
    print(f"\n  ⚠️  Sample size may be insufficient.")

  POWER ANALYSIS
  Effect size (Cohen's h) : 0.0878
  Required sample size    : 2,038 per group
  Actual sample size      : 12,500 per group

  ✅ Our sample size is SUFFICIENT.
  We had 12,500 users vs the 2,038 required.
  Our test has adequate statistical power.


In [32]:
# Check if the nudge hurt average order value (guardrail metric)
aov_control   = control[control['purchased']==1]['order_value_inr'].mean()
aov_treatment = treatment[treatment['purchased']==1]['order_value_inr'].mean()

# T-test on order values
t_stat, p_aov = stats.ttest_ind(
    control[control['purchased']==1]['order_value_inr'],
    treatment[treatment['purchased']==1]['order_value_inr']
)

print(f"{'='*50}")
print(f"  GUARDRAIL METRIC — Avg Order Value")
print(f"{'='*50}")
print(f"  Control AOV   : ₹{aov_control:,.0f}")
print(f"  Treatment AOV : ₹{aov_treatment:,.0f}")
print(f"  Difference    : ₹{aov_treatment - aov_control:+,.0f}")
print(f"  P-value       : {p_aov:.4f}")
print(f"{'='*50}")

if p_aov > 0.05:
    print(f"\n  ✅ AOV difference is NOT significant (p={p_aov:.4f})")
    print(f"  The nudge did NOT hurt order values.")
    print(f"  Safe to roll out without revenue quality risk.")
else:
    print(f"\n  ⚠️  AOV difference IS significant (p={p_aov:.4f})")
    print(f"  Investigate before full rollout.")

  GUARDRAIL METRIC — Avg Order Value
  Control AOV   : ₹857
  Treatment AOV : ₹862
  Difference    : ₹+5
  P-value       : 0.5272

  ✅ AOV difference is NOT significant (p=0.5272)
  The nudge did NOT hurt order values.
  Safe to roll out without revenue quality risk.


In [33]:
print(f"""
{'='*55}
  EXPERIMENT CONCLUSION — UPI NUDGE A/B TEST
{'='*55}

  Experiment period  : Jan 2024 – Jun 2024
  Total users        : 25,000 (12,500 per group)

  PRIMARY METRIC
  Control conversion rate   : {cr_control:.2%}
  Treatment conversion rate : {cr_treatment:.2%}
  Absolute lift             : +{absolute_lift:.2%}
  Relative lift             : +{relative_lift:.2%}

  STATISTICAL VALIDITY
  Chi-square p-value        : {p_value:.6f}
  Result                    : {'Significant ✅' if p_value < 0.05 else 'Not Significant ❌'}
  Sample size adequate      : {'Yes ✅' if n_control >= required_n else 'No ❌'}

  GUARDRAIL CHECK
  AOV impact                : ₹{aov_treatment - aov_control:+,.0f} (p={p_aov:.4f})
  Result                    : {'No harm detected ✅' if p_aov > 0.05 else 'Investigate ⚠️'}

  RECOMMENDATION
  → Roll out UPI nudge to 100% of users
  → Prioritise mobile and Tier 2/3 city traffic
  → Expected revenue uplift: ₹3.6L/month at scale

{'='*55}
""")


  EXPERIMENT CONCLUSION — UPI NUDGE A/B TEST

  Experiment period  : Jan 2024 – Jun 2024
  Total users        : 25,000 (12,500 per group)

  PRIMARY METRIC
  Control conversion rate   : 15.00%
  Treatment conversion rate : 18.26%
  Absolute lift             : +3.26%
  Relative lift             : +21.76%

  STATISTICAL VALIDITY
  Chi-square p-value        : 0.000000
  Result                    : Significant ✅
  Sample size adequate      : Yes ✅

  GUARDRAIL CHECK
  AOV impact                : ₹+5 (p=0.5272)
  Result                    : No harm detected ✅

  RECOMMENDATION
  → Roll out UPI nudge to 100% of users
  → Prioritise mobile and Tier 2/3 city traffic
  → Expected revenue uplift: ₹3.6L/month at scale


